### **Fine Tuning**

#### **What is Fine Tuning ?**

Fine Tuning is the process of taking an already trained model and training it further on domain-specific or task-specific data so that it performs better for a particular use case. 

The key insight is that training a large language model from scratch is extremely difficult (massive compute, massive data, massive time). Instead, we leverage the general knowledge already embedded in a publicly-available open-source model, and specialise it for our needs.

**Analogy** : Rather than teaching each and everything to a person and then make them a special chef, you take a general chef who generally knows how to cook, you adjust their skills for a specific type of cuisine and people. 

**A day in the Life of an AI engineer** : Under 99% cases, people donot do fine-tuning, but the understanding on how-to is important. 

#### **The LLM training Pipeline**

Every production LLM goes through (at most) 3 sequential training stages. Understanding these is the foundation for understanding fine-tuning.

| Stage | HuggingFace model ID | What it knows |
|---|---|---|
| Pre-trained (raw) | `meta-llama/Meta-Llama-3-8B` | Next token prediction, broad world knowledge, no instruction-following |
| Instruction-tuned (SFT) | `meta-llama/Meta-Llama-3-8B-Instruct` | How to follow instructions, conversation, Q&A format |
| Preference-aligned (RLHF) | `OpenRLHF/Llama-3-8b-rlhf-100k` | How to respond in a way humans prefer; aligned to human values |


**You can pick the model from any of these stages and apply your own custom fine-tuning on top of it. The right choice depends on your use case**

1) **Pre-training (raw model)**

- Trained on internet-scale raw data and the process is also called Unsupervised/Self-supervised learning. 
- Basis this, the model learns to predict the next token (using decoder-only architecture). It acquires the broad world knowledge but has no idea how to follow the instructions. 
- Having a Pre-trained model is like just knowing the language.


2) **Supervised Fine Tuning (SFT) / Instruction Fine tuning** 

- Trained on instruction dataset which has a paired input → output data so the model learns to follow the instructions. 
- Meta collected instruction datasets, presented them as (input, output) pairs, and fine-tuned the raw model. This produces the "Instruct" variant. The model now knows how to have a conversation, answer questions, and follow user prompts.


3) **Preference Allignment (optional but increasingly common)** : 

- The model is further trained to respond according to human preferences. 
- Human annotators label which of two model responses they prefer. That preference data is used to retrain the model. Not mandatory — but large companies (OpenAI, Anthropic, Google) all do this now. Introduced prominently by ChatGPT / OpenAI.

#### **Choosing the right base model for custom fine-tuning**

The choice of starting stage/model depends entirely on what output you want from your fine-tuned model. 

**Use case**: A pharma company has internal documents on molecular studies and wants an AI model to work with that domain-specific data.

**Solution** : Take an open-source model (say `meta-llama/Meta-Llama-3-8B`), download it to your server and then fine-tune it as training a model from scratch is very expensive. 

**One important point in the ask was to generate the data and not build a conversational-AI, so you do not need an Instruct level model as it is tuned to follow input output structures.**


| Use case | Goal | Starting model |
|---|---|---|
| Pure text / data generation | Generate new pharma text (not Q&A). | `meta-llama/Meta-Llama-3-8B` |
| Chatbot / Q&A / conversational AI | Question-answering system over pharma data. | `meta-llama/Meta-Llama-3-8B-Instruct`



**The key point**

The instruction-tuned model already knows how to follow instructions in general. Your custom SFT then teaches it pharma-specific instructions. This is almost always the better starting point for chatbot use cases.

**Practical Tip**

Training an LLM from scratch is extremely difficult and expensive. Always start from an open-source base model and fine-tune it. Companies like Meta, Mistral, Alibaba, and Google make high-quality base models freely available.

#### **Parameter-level understanding**

To understand fine-tuning deeply, you need to undersatdn what is actually being updated inside the model. 

**What is a parameter?**

A parameter = weights and biases of the neural network. In a transformer-based LLM, parameters exist in two main places:

- **Self attention layer**: Contains the Query (Q), Key (K), and Value (V) weight matrices. These are learned during training.
- **Feed Forward Neural Network** : Contains its own weights and biases. Also trained during the learning process.

When we "fine-tune" a model, we are retraining some or all of these weights on new data. The question is: which ones?

#### **Approaches of Fine Tuning**

**Full fine-tuning vs partial fine-tuning**

| Type | What gets updated | Cost | Practical? |
|---|---|---|---|
| Full fine-tuning | All parameters (weights & biases) | Huge GPU power + massive infrastructure | No - we will NOT do this |
| Partial fine-tuning | A subset of parameters | Manageable; can run on a single GPU | Yes - this is our focus |


**Full fine-tuning is what companies like Meta do when building the base model. For custom/domain fine-tuning, we always use partial fine-tuning.**

##### **Partial fine-tuning (Supervised Fine tuning)**

1) **Old-school method (Layer Freezing)** :

Before PEFT, the standard technique for partial fine-tuning was layer freezing. This was used extensively with CNNs (VGG, ResNet) and early LLMs (BERT, BART, T5).

**1A) Freeze all layers → train only the last output layer**

- All existing weights are frozen (not updated). 
- Only the final output layer's weights are retrained. 
- Effective for simple task adaptation but very limited.

**1B) Freeze early layers → retrain remaining last layers**

- The first N layers (which learn generic, low-level representations) are frozen. 
- The later layers (which encode more task-specific knowledge) are retrained. 
- More powerful than Method 1.

**Why This fails for latest LLMs**

Modern LLMs (Llama, Mistral, GPT-4 class) have enormous architectures — billions of parameters, dozens or hundreds of layers. Even "partial" retraining of the last few layers requires massive infrastructure. The old-school method does not scale.
The old-school method works fine for CNN architectures and smaller LLMs like BERT. For billion-parameter models, we need a completely different approach: PEFT.

2) **PEFT (Parameter Efficient Fine Tuning)** : It is the umbrella term for a family of modern techniques that allow fine-tuning large models using only a subset of parameters i.e. smaller matrices. Basis PEFT, we can do model training on a single GPU and small RAM. 

The techniques to do PEFT are : 

- LoRA: Low Rank adaption. This is the industry standard. It attaches a small adapter to the model. 
- QLoRA : Quantised Low Rank adaption. This is actually LoRA applied to a quantised (compressed) model. Reduces memory even further. 
- DoRA : Weight decomposition Low Rank adaptation. A variant of LoRA that applies weight decomposition. More optimised, latest technique.
- BitFit : Only trains the bias terms. Extremely lightweight but limited.
- IA3 : Based on attention mechanism modification. 

##### **Preference Alignment**

After SFT, a model knows how to follow instructions — but it may not respond the way humans prefer. Preference alignment is Stage 3 of the pipeline: training the model to match human preference.

**How is preference data collected?**

In typical preference-collection flows (similar to ChatGPT-style interfaces), users are shown two responses to the same prompt and asked "Which response do you prefer?" This generates preference-annotated data — each sample records which response was chosen and which was rejected. The data format for preference tuning includes: a prompt, a "chosen" response (preferred by humans), and a "rejected" response (not preferred). This is sometimes called a triplet: (prompt, chosen, rejected).

**Is it mandatory?**

Preference alignment is not mandatory every time, but it is now standard practice at large AI companies. It makes models safer, more helpful, and more aligned with human values. OpenAI introduced it with ChatGPT; it has since been adopted by Anthropic, Google, Meta, and others.

**Preference Alignment algorithms**
- RLHF : Reinforcement Learning from Human Feedback is a framework, not a single algorithm. It uses reinforcement learning — specifically the Deep RL branch — and the underlying optimisation algorithm is PPO (Proximal Policy Optimisation). The model is treated as a "policy" that is updated to maximise a reward signal derived from human preference judgements. 
- GRPO : Group Relative Policy Optimisation. An updated variant of RLHF / PPO. Used to train DeepSeek models. Understanding RLHF first will make GRPO straightforward. 
- DPO : Direct Preference Optimisation. DPO removes the need for a separate reward model and reinforcement learning. It directly optimises the model on (chosen, rejected) pairs using a cross-entropy-style objective. Simpler and often more stable than RLHF. 
- ORPO : Odds Ratio Preference Optimisation. The latest update to DPO. More efficient and effective. 

#### **Fine-Tuning Frameworks**

For Finetuning we use 2 frameworks :
- HuggingFace : The primary ecosystem for fine-tuning. The transformers library provides model loading; peft provides LoRA/QLoRA; trl (Transformer Reinforcement Learning) provides SFT trainers, DPO trainers, and RLHF utilities. It is the foundation of the fine-tuning ecosystem. Older, Opensource, Market Leader. 
- Unsloth : A newer, highly optimised fine-tuning library built on top of HuggingFace foundations that makes LoRA and QLoRA 2–5× faster with lower VRAM usage. Excellent for free-tier Google Colab. This enables : 
    - Faster training
    - Lower VRAM usage
    - Longer context training
    - Cheaper fine-tuning

There are a few other names again built up on top of HF itself like Llamafactory, Axolotal. 

All fine-tuning works should be done via Google Colab — you do not need a local GPU. If you have at least 16 GB of RAM locally, you can experiment with Ollama-based models, but training/fine-tuning should be done on Colab.